In [2]:
import pandas as pd

In [3]:
healthy_data = pd.read_csv('data/healthy.csv')
healthy_data.dropna(inplace=True) #drop all NaN values

pathological_data = pd.read_csv('data/deduplicated_pathological.csv')
pathological_data.dropna(inplace=True) #drop all NaN values

## Phrase data

In [22]:
#Select only phrases
healthy_phrase = healthy_data[healthy_data['recording'] == 'phrase']
pathological_phrase = pathological_data[pathological_data['recording'] == 'phrase']
print(len(healthy_phrase))
print(len(pathological_phrase))

634
1354


In [27]:
#remove pathologies with less than 20 recordings
counts = pathological_phrase['type'].value_counts().to_dict()
for k, v in counts.items():
    if v < 20:
        #print(k)
        pathological_phrase = pathological_phrase[pathological_phrase['type'] != k]

In [30]:
#balance data
pathological_phrase['_rank'] = pathological_phrase.groupby('type').cumcount()

# Sort by rank so lower-rank items from all groups come first
# This caps frequent values while preserving all rare values
pathological_phrase_shrunk = (
    pathological_phrase.sort_values('_rank')
    .head(len(healthy_phrase))
    .drop(columns=['_rank'])
)

#print(len(pathological_phrase_shrunk))
#print(pathological_phrase_shrunk['type'].value_counts().to_dict())

634
{'Hyperfunktionelle Dysphonie': 52, 'Chordektomie': 51, 'Laryngitis': 51, 'Psychogene Dysphonie': 51, 'Rekurrensparese': 51, 'Funktionelle Dysphonie': 51, 'Spasmodische Dysphonie': 51, 'Dysphonie': 51, 'Kontaktpachydermie': 49, 'Reinke Odem': 40, 'Dysodie': 31, 'Stimmlippenpolyp': 30, 'Frontolaterale Teilresektion': 30, 'Vox senilis': 25, 'Stimmlippenkarzinom': 20}


In [32]:
#save to a file
combined_phrase = pd.concat([healthy_phrase, pathological_phrase_shrunk], ignore_index=True)
combined_phrase.to_csv("data/preped/phrase.csv", index=False)

## Normal pitch vowels data

## All pitches vowel data

In [ ]:
#Remove pathologies with less than 10 patients
print("="*50)
deduplicate_count = pathological_phrase.groupby('type')['person_id'].count().to_dict()
print(f"Number of pathologies before removal: {len(deduplicate_count)}")

for pathology, number in deduplicate_count.items():
    #print(pathology, number)
    if number<10:
        pathological_phrase.drop(pathological_phrase[pathological_phrase['type'] == pathology].index, inplace=True)

after_removal = pathological_phrase.groupby('type')['person_id'].count().to_dict()
print(f"Number of pathologies after removal: {len(after_removal)}")

#Now balance data